## 1. Imports


In [ ]:
import time
import warnings

from loguru import logger

warnings.filterwarnings("ignore")

from mlops_aml_transactions.config import RAW_AML_DEFAULT_FILES
from mlops_aml_transactions.data.raw import load_raw_transaction_files
from mlops_aml_transactions.eda_plotly import (
    column_summary,
    data_overview,
    payment_currency_stats,
    payment_format_stats,
    plot_class_balance,
    plot_daily_laundering_rate,
    plot_hourly_laundering_rate,
    plot_log_amount_paid_distribution,
)
from mlops_aml_transactions.features_lgb import engineer_features_lgb
from mlops_aml_transactions.modeling.lgb_aml import train_lgbm_aml


## 2. Data Loading


In [ ]:
df_raw = load_raw_transaction_files(RAW_AML_DEFAULT_FILES)
df_raw.head()


## 3. Exploratory Data Analysis


In [ ]:
overview = data_overview(df_raw)
overview


In [ ]:
column_summary(df_raw)


In [ ]:
fig = plot_class_balance(df_raw)
fig.show()


### Что показывает график `Class Balance`

- Этот график сравнивает количество обычных транзакций (`Normal`) и подозрительных (`Laundering`).
- Ось `Y` здесь логарифмическая, поэтому разница между классами визуально сжата. Это сделано специально: без логарифмической шкалы столбец подозрительных операций был бы почти не виден.
- Главный вывод: в данных очень сильный дисбаланс классов. Подозрительных операций крайне мало, поэтому модель нельзя оценивать только по `accuracy` - важнее смотреть на `recall`, `precision`, `PR-AUC` и порог классификации.


In [ ]:
fig = plot_daily_laundering_rate(df_raw)
fig.show()


### Что показывает график `Daily Laundering Rate`

- Здесь по оси `X` идут дни, а по оси `Y` - доля подозрительных транзакций в этот день.
- График нужен, чтобы понять, равномерно ли распределены подозрительные операции по времени или есть отдельные дни со всплесками.
- Если на линии видны пики, это может означать, что в некоторые дни поведение потока заметно меняется. Для модели это сигнал, что временные признаки (`day`, `hour`, `weekday`) действительно могут быть полезны.


In [ ]:
fig = plot_hourly_laundering_rate(df_raw)
fig.show()


### Что показывает график `Hourly Laundering Rate`

- Этот график показывает, как меняется доля подозрительных операций в зависимости от часа суток.
- Он помогает увидеть, есть ли у подозрительных транзакций характерные временные окна, например ночные часы или периоды низкой обычной активности.
- Если в определённые часы доля `laundering` выше, это подтверждает полезность признаков `hour` и `is_night` в модели.


In [ ]:
fig = plot_log_amount_paid_distribution(df_raw)
fig.show()


### Что показывает график `Distribution of log(Amount Paid + 1)`

- На этом графике сравнивается распределение сумм платежей для обычных и подозрительных транзакций.
- Используется преобразование `log(Amount Paid + 1)`, чтобы очень большие суммы не "раздавили" весь график и можно было сравнивать формы распределений.
- Если кривые для `Normal` и `Laundering` заметно различаются, значит размер платежа может быть полезным сигналом для модели. Если они сильно перекрываются, одной только суммы недостаточно и нужно опираться на комбинацию признаков.


In [ ]:
payment_format_stats(df_raw).head(15)


In [ ]:
payment_currency_stats(df_raw).head(15)


## 4. Feature Engineering & Training

Реализация в пакете: `mlops_aml_transactions.features_lgb` (признаки, порог Fβ), `mlops_aml_transactions.modeling.lgb_aml` (LightGBM, OOF, MLflow, SHAP).


### Время выполнения


In [ ]:
started_at = time.time()

df_features = engineer_features_lgb(df_raw)
model, metrics = train_lgbm_aml(df_features)

total_minutes = (time.time() - started_at) / 60
logger.info("Done in {:.1f} min", total_minutes)
metrics
